# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.28 — FAST
## Full GVH EOM Derivation and Variational-Closure Gate Audit

### Mission

Ouvrir la première dérivation EOM explicite à partir de l'action candidate déjà matérialisée en `.3.3.24–.3.3.25`, tout en respectant le verrou P/D établi en `.3.3.27.3–.3.3.27.4`.

La cible est :

\[
\boxed{
\frac{\delta S_{\rm cand}}
{\delta(g^{\mu\nu},u^\mu,e_{(A)}^\mu)}
}
\]

avec trois exigences :

1. **ne pas** déclarer dérivés les paramètres cinématiques \(\omega_{10},\omega_{21},u_2\) tant qu'aucune solution réduite des EOM ne les sélectionne ;
2. distinguer une **équation variationnelle exacte** d'une **expansion complète en composantes** ;
3. conserver les quatre niveaux :
   - Niveau 1 — GVH ;
   - Niveau 2 — physique établie comme benchmark ;
   - Niveau 3 — témoin numérique minimal ;
   - Niveau 4 — verdict scientifique.

### Provenance canonique

Upstream immédiat exécuté `.3.3.27.4` :

`b6847b27383c4c460450dd6613a9cf47369b26cd9ae4b135b169ec63a6031b36`

Supports structurels :

- `.3.3.24` exécuté : `7286f9ba6558a11b6160b4e6059352150572f636aac239578e51f5efd543f448`
- `.3.3.25` exécuté : `cbf6e3009a1e8757b3b04f9a2b6d7415acb5e91bdb824db62137022dca460e00`

Les références `O-14`, `O-29`, `O-31` sont conservées comme renvois OPEN du catalogue consolidé signalés par l'audit protocolaire ; ce notebook n'en redéfinit pas la sémantique.

In [1]:
from __future__ import annotations
import json, sys, math
from pathlib import Path
import numpy as np
import pandas as pd
import sympy as sp

UPSTREAM = {
    "p33274": {
        "sha256": "b6847b27383c4c460450dd6613a9cf47369b26cd9ae4b135b169ec63a6031b36",
        "size_bytes": 33636,
        "local_audit_pass": True,
        "SI_bridge_materialized": True,
        "absolute_SI_identifiable": False,
        "new_GVH_physics_validated": False,
    },
    "p3324": {
        "sha256": "7286f9ba6558a11b6160b4e6059352150572f636aac239578e51f5efd543f448",
        "size_bytes": 26259,
        "cubic_triad_candidate_materialized": True,
    },
    "p3325": {
        "sha256": "cbf6e3009a1e8757b3b04f9a2b6d7415acb5e91bdb824db62137022dca460e00",
        "size_bytes": 26989,
        "triad_constraint_algebra_audited": True,
        "triad_propagating_dof": 0,
        "onshell_distinctness_established": False,
    },
}

G28_UPSTREAM_PROVENANCE_PASS = all([
    UPSTREAM["p33274"]["local_audit_pass"],
    UPSTREAM["p33274"]["SI_bridge_materialized"],
    not UPSTREAM["p33274"]["absolute_SI_identifiable"],
    not UPSTREAM["p33274"]["new_GVH_physics_validated"],
    UPSTREAM["p3324"]["cubic_triad_candidate_materialized"],
    UPSTREAM["p3325"]["triad_constraint_algebra_audited"],
    UPSTREAM["p3325"]["triad_propagating_dof"] == 0,
    not UPSTREAM["p3325"]["onshell_distinctness_established"],
])

G28_BLOCKED_OPEN_REFS = ["O-14", "O-29", "O-31"]

assert G28_UPSTREAM_PROVENANCE_PASS
print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("G28_UPSTREAM_PROVENANCE_PASS =", G28_UPSTREAM_PROVENANCE_PASS)
print("G28_BLOCKED_OPEN_REFS =", G28_BLOCKED_OPEN_REFS)

Python = 3.13.15
SymPy = 1.14.0
G28_UPSTREAM_PROVENANCE_PASS = True
G28_BLOCKED_OPEN_REFS = ['O-14', 'O-29', 'O-31']


# Niveau 1 — GVH

## 1.1 Action candidate contrainte

On adopte la signature \((-+++)\) et :

\[
u^\mu u_\mu=-1,
\qquad
h_{\mu\nu}=g_{\mu\nu}+u_\mu u_\nu.
\]

Le secteur vectoriel déjà audité est :

\[
\mathcal L_u=
-c_1(\nabla_\mu u_\nu)(\nabla^\mu u^\nu)
-c_2(\nabla_\mu u^\mu)^2
-c_3(\nabla_\mu u_\nu)(\nabla^\nu u^\mu)
+c_4 a_\mu a^\mu,
\]

\[
a^\mu=u^\nu\nabla_\nu u^\mu.
\]

Le trièdre cubique vérifie :

\[
u_\mu e_{(A)}^\mu=0,
\qquad
g_{\mu\nu}e_{(A)}^\mu e_{(B)}^\nu=\delta_{AB}.
\]

On définit :

\[
C_{\mu\nu\rho\sigma}
=
\sum_{A=1}^{3}
e^{(A)}_\mu e^{(A)}_\nu e^{(A)}_\rho e^{(A)}_\sigma,
\]

\[
\mathcal O_C
=
C^{\mu\nu\rho\sigma}\sigma_{\mu\nu}\sigma_{\rho\sigma}.
\]

L'action de travail est :

\[
\boxed{
S_{\rm cand}
=
\frac{1}{16\pi G_0}
\int d^4x\sqrt{-g}
\left[
R+\mathcal L_u
+\lambda_u(u^2+1)
+\frac{\zeta_C}{2}\mathcal O_C
+\sum_A\eta_A\,u\!\cdot\!e_A
+\frac12\sum_{A,B}\eta_{AB}
(e_A\!\cdot\!e_B-\delta_{AB})
\right].
}
\]

\(\eta_{AB}=\eta_{BA}\).

Cette action **n'ajoute toujours aucun terme cinétique \(\nabla e_A\)** : le trièdre reste auxiliaire dans le candidat actuel.

In [2]:
# P/D ledger of the action itself
action_ledger = pd.DataFrame([
    {"object":"g_mn", "role":"DYNAMICAL_FIELD", "status":"VARIED", "EOM_expected":True},
    {"object":"u^m", "role":"DYNAMICAL_FIELD", "status":"VARIED", "EOM_expected":True},
    {"object":"e_A^m", "role":"AUXILIARY_DIRECTIONAL_FIELD", "status":"VARIED", "EOM_expected":True},
    {"object":"c1,c2,c3,c4", "role":"THEORY_COUPLINGS", "status":"PRESCRIBED_THEORY_PARAMETERS", "EOM_expected":False},
    {"object":"zeta_C", "role":"CUBIC_COUPLING", "status":"PRESCRIBED_THEORY_PARAMETER", "EOM_expected":False},
    {"object":"G0", "role":"GRAVITATIONAL_SCALE", "status":"PRESCRIBED_OR_INDEPENDENTLY_CALIBRATED", "EOM_expected":False},
    {"object":"lambda_u", "role":"UNIT_CONSTRAINT_MULTIPLIER", "status":"ALGEBRAICALLY_DERIVABLE", "EOM_expected":True},
    {"object":"eta_A", "role":"ORTHOGONALITY_MULTIPLIER", "status":"ALGEBRAICALLY_DERIVABLE", "EOM_expected":True},
    {"object":"eta_AB", "role":"ORTHONORMALITY_MULTIPLIER", "status":"ALGEBRAICALLY_DERIVABLE", "EOM_expected":True},
])

G28_CONSTRAINED_ACTION_MATERIALIZED = len(action_ledger) == 9
G28_TRIAD_OWN_KINETIC_TERM_PRESENT = False
G28_ACTION_SCOPE_EXPLICIT = True

assert G28_CONSTRAINED_ACTION_MATERIALIZED
assert not G28_TRIAD_OWN_KINETIC_TERM_PRESENT

print(action_ledger.to_string(index=False))
print("G28_CONSTRAINED_ACTION_MATERIALIZED =", G28_CONSTRAINED_ACTION_MATERIALIZED)
print("G28_TRIAD_OWN_KINETIC_TERM_PRESENT =", G28_TRIAD_OWN_KINETIC_TERM_PRESENT)

     object                        role                                 status  EOM_expected
       g_mn             DYNAMICAL_FIELD                                 VARIED          True
        u^m             DYNAMICAL_FIELD                                 VARIED          True
      e_A^m AUXILIARY_DIRECTIONAL_FIELD                                 VARIED          True
c1,c2,c3,c4            THEORY_COUPLINGS           PRESCRIBED_THEORY_PARAMETERS         False
     zeta_C              CUBIC_COUPLING            PRESCRIBED_THEORY_PARAMETER         False
         G0         GRAVITATIONAL_SCALE PRESCRIBED_OR_INDEPENDENTLY_CALIBRATED         False
   lambda_u  UNIT_CONSTRAINT_MULTIPLIER                ALGEBRAICALLY_DERIVABLE          True
      eta_A    ORTHOGONALITY_MULTIPLIER                ALGEBRAICALLY_DERIVABLE          True
     eta_AB   ORTHONORMALITY_MULTIPLIER                ALGEBRAICALLY_DERIVABLE          True
G28_CONSTRAINED_ACTION_MATERIALIZED = True
G28_TRIAD_OWN_KINETIC_TERM_

## 1.2 Décomposition du secteur cubique

Introduisons la matrice du cisaillement dans le trièdre :

\[
M_{AB}
=
e_{(A)}^\mu\sigma_{\mu\nu}e_{(B)}^\nu
=
M_{BA}.
\]

Alors :

\[
\boxed{
\mathcal O_C
=
\sum_A M_{AA}^2.
}
\]

Pour une rotation infinitésimale du trièdre dans le plan \(AB\),

\[
e_A\rightarrow e_A+\epsilon_{AB}e_B,
\qquad
\epsilon_{AB}=-\epsilon_{BA},
\]

la variation de \(\mathcal O_C\) doit donner l'équation d'orientation du trièdre.

In [3]:
m11,m22,m33,m12,m13,m23 = sp.symbols(
    "m11 m22 m33 m12 m13 m23", real=True
)
a,b,c = sp.symbols("a b c", real=True)

M = sp.Matrix([
    [m11,m12,m13],
    [m12,m22,m23],
    [m13,m23,m33],
])

R12 = sp.Matrix([
    [sp.cos(a),-sp.sin(a),0],
    [sp.sin(a), sp.cos(a),0],
    [0,0,1],
])
R13 = sp.Matrix([
    [sp.cos(b),0,-sp.sin(b)],
    [0,1,0],
    [sp.sin(b),0,sp.cos(b)],
])
R23 = sp.Matrix([
    [1,0,0],
    [0,sp.cos(c),-sp.sin(c)],
    [0,sp.sin(c), sp.cos(c)],
])

def OC_of_rotation(R):
    Mr = sp.simplify(R.T*M*R)
    return sp.expand(sum(Mr[i,i]**2 for i in range(3)))

g12 = sp.simplify(sp.diff(OC_of_rotation(R12), a).subs(a,0))
g13 = sp.simplify(sp.diff(OC_of_rotation(R13), b).subs(b,0))
g23 = sp.simplify(sp.diff(OC_of_rotation(R23), c).subs(c,0))

expected = [
    4*m12*(m11-m22),
    4*m13*(m11-m33),
    4*m23*(m22-m33),
]

G28_ORIENTATION_FIRST_VARIATION_EXACT_PASS = all(
    sp.simplify(x-y) == 0
    for x,y in zip([g12,g13,g23], expected)
)

assert G28_ORIENTATION_FIRST_VARIATION_EXACT_PASS

print("dO/dtheta12 =", sp.factor(g12))
print("dO/dtheta13 =", sp.factor(g13))
print("dO/dtheta23 =", sp.factor(g23))
print("G28_ORIENTATION_FIRST_VARIATION_EXACT_PASS =",
      G28_ORIENTATION_FIRST_VARIATION_EXACT_PASS)

dO/dtheta12 = 4*m12*(m11 - m22)
dO/dtheta13 = 4*m13*(m11 - m33)
dO/dtheta23 = 4*m23*(m22 - m33)
G28_ORIENTATION_FIRST_VARIATION_EXACT_PASS = True


## 1.3 EOM explicite du trièdre \(e_{(A)}^\mu\)

Posons :

\[
s_A
=
e_A^\mu e_A^\nu\sigma_{\mu\nu}
=
M_{AA}.
\]

À \(g_{\mu\nu}\) et \(u^\mu\) fixés :

\[
\delta_{e_A}\mathcal O_C
=
4s_A\sigma_{\kappa\nu}e_A^\nu\,\delta e_A^\kappa.
\]

L'équation d'Euler du trièdre est donc :

\[
\boxed{
\mathcal E^{(e_A)}_\kappa
=
2\zeta_C s_A\sigma_{\kappa\nu}e_A^\nu
+\eta_Au_\kappa
+\sum_B\eta_{AB}e_{B\kappa}
=0.
}
\]

La projection sur \(u^\kappa\) détermine la partie \(\eta_A\), tandis que la projection sur \(e_B^\kappa\) donne :

\[
\eta_{AB}
=
-2\zeta_C s_A M_{AB}.
\]

Mais \(\eta_{AB}=\eta_{BA}\). Il faut donc :

\[
\boxed{
\zeta_C(s_A-s_B)M_{AB}=0,
\qquad A<B.
}
\]

C'est la première EOM Diagonal-Cubic explicitement dérivée dans ce segment : elle sélectionne des **branches d'orientation**, pas encore des valeurs numériques de \(\omega_{10},\omega_{21},u_2\).

In [4]:
zeta = sp.symbols("zeta_C", nonzero=True, real=True)
s1,s2,s3 = sp.symbols("s1 s2 s3", real=True)
M12,M13,M23 = sp.symbols("M12 M13 M23", real=True)

# Symmetry of eta_AB after projecting the triad EOM:
eta12_from_A = -2*zeta*s1*M12
eta21_from_B = -2*zeta*s2*M12
eta13_from_A = -2*zeta*s1*M13
eta31_from_B = -2*zeta*s3*M13
eta23_from_A = -2*zeta*s2*M23
eta32_from_B = -2*zeta*s3*M23

residuals = [
    sp.factor(eta12_from_A-eta21_from_B),
    sp.factor(eta13_from_A-eta31_from_B),
    sp.factor(eta23_from_A-eta32_from_B),
]
expected_residuals = [
    -2*zeta*(s1-s2)*M12,
    -2*zeta*(s1-s3)*M13,
    -2*zeta*(s2-s3)*M23,
]

G28_TRIAD_EOM_PROJECTED_EXACT_PASS = all(
    sp.simplify(x-y) == 0
    for x,y in zip(residuals, expected_residuals)
)

G28_TRIAD_ORIENTATION_EOM_EXPLICIT = G28_TRIAD_EOM_PROJECTED_EXACT_PASS

assert G28_TRIAD_EOM_PROJECTED_EXACT_PASS

print("eta symmetry residuals =", residuals)
print("G28_TRIAD_EOM_PROJECTED_EXACT_PASS =", G28_TRIAD_EOM_PROJECTED_EXACT_PASS)
print("G28_TRIAD_ORIENTATION_EOM_EXPLICIT =", G28_TRIAD_ORIENTATION_EOM_EXPLICIT)

eta symmetry residuals = [-2*M12*zeta_C*(s1 - s2), -2*M13*zeta_C*(s1 - s3), -2*M23*zeta_C*(s2 - s3)]
G28_TRIAD_EOM_PROJECTED_EXACT_PASS = True
G28_TRIAD_ORIENTATION_EOM_EXPLICIT = True


## 1.4 EOM du champ \(u^\mu\)

Écrivons le secteur Einstein-æther sous la forme :

\[
\mathcal L_u
=
-K^{\alpha\beta}{}_{\mu\nu}
\nabla_\alpha u^\mu
\nabla_\beta u^\nu,
\]

\[
K^{\alpha\beta}{}_{\mu\nu}
=
c_1g^{\alpha\beta}g_{\mu\nu}
+c_2\delta^\alpha_\mu\delta^\beta_\nu
+c_3\delta^\alpha_\nu\delta^\beta_\mu
-c_4u^\alpha u^\beta g_{\mu\nu}.
\]

Définissons :

\[
J^\alpha{}_\mu
=
K^{\alpha\beta}{}_{\mu\nu}\nabla_\beta u^\nu.
\]

Pour le secteur cubique, introduisons le projecteur STF spatial :

\[
\Delta_{\rho\sigma}{}^{\alpha\beta}
=
h_{(\rho}{}^\alpha h_{\sigma)}{}^\beta
-\frac13h_{\rho\sigma}h^{\alpha\beta},
\]

\[
\sigma_{\rho\sigma}
=
\Delta_{\rho\sigma}{}^{\alpha\beta}
\nabla_\alpha u_\beta,
\]

et :

\[
\Pi^{\rho\sigma}
=
\sum_A s_A e_A^\rho e_A^\sigma.
\]

Le courant principal cubique est :

\[
Q^\alpha{}_\mu
=
\Pi^{\rho\sigma}
\Delta_{\rho\sigma}{}^{\alpha\beta}
g_{\beta\mu}.
\]

La dépendance algébrique du projecteur en \(u^\mu\) donne exactement :

\[
\mathcal A^{(C)}_\mu
=
\Pi^{\rho\sigma}
\frac{\partial
\Delta_{\rho\sigma}{}^{\alpha\beta}}
{\partial u^\mu}
\nabla_\alpha u_\beta.
\]

Dans la normalisation brute de l'action ci-dessus :

\[
\boxed{
2\left[
\nabla_\alpha J^\alpha{}_\mu
+c_4 a_\alpha\nabla_\mu u^\alpha
\right]
+\zeta_C
\left[
-\nabla_\alpha Q^\alpha{}_\mu
+\mathcal A^{(C)}_\mu
\right]
+2\lambda_u u_\mu
+\sum_A\eta_A e_{A\mu}
=0.
}
\]

Cette forme est une dérivée fonctionnelle exacte. L'expansion composante complète de \(\mathcal A^{(C)}_\mu\) reste à matérialiser séparément avant toute réduction forte.

In [5]:
# Structural audit of the u-EOM.
# We do not replace an exact functional object by an invented component expansion.

u_eom_terms = pd.DataFrame([
    {"term":"2 nabla_alpha J^alpha_mu", "origin":"EA derivative current", "status":"EXPLICIT_STANDARD_FORM"},
    {"term":"2 c4 a_alpha nabla_mu u^alpha", "origin":"u-dependence of K", "status":"EXPLICIT_STANDARD_FORM"},
    {"term":"- zeta_C nabla_alpha Q^alpha_mu", "origin":"cubic principal variation", "status":"EXPLICIT_DERIVED_FORM"},
    {"term":"zeta_C A_C_mu", "origin":"u-dependence of STF projector", "status":"EXACT_FUNCTIONAL_DEFINITION"},
    {"term":"2 lambda_u u_mu", "origin":"unit constraint", "status":"EXPLICIT"},
    {"term":"sum_A eta_A e_A_mu", "origin":"u.e_A constraints", "status":"EXPLICIT"},
])

G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED = len(u_eom_terms) == 6
G28_CUBIC_VECTOR_PRINCIPAL_CURRENT_MATERIALIZED = True
G28_CUBIC_VECTOR_ALGEBRAIC_REMAINDER_FUNCTIONALLY_EXACT = True
G28_CUBIC_VECTOR_ALGEBRAIC_REMAINDER_COMPONENT_EXPANDED = False

assert G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED
assert G28_CUBIC_VECTOR_PRINCIPAL_CURRENT_MATERIALIZED
assert G28_CUBIC_VECTOR_ALGEBRAIC_REMAINDER_FUNCTIONALLY_EXACT

print(u_eom_terms.to_string(index=False))
print("G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED =",
      G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED)
print("G28_CUBIC_VECTOR_ALGEBRAIC_REMAINDER_COMPONENT_EXPANDED =",
      G28_CUBIC_VECTOR_ALGEBRAIC_REMAINDER_COMPONENT_EXPANDED)

                           term                        origin                      status
       2 nabla_alpha J^alpha_mu         EA derivative current      EXPLICIT_STANDARD_FORM
  2 c4 a_alpha nabla_mu u^alpha             u-dependence of K      EXPLICIT_STANDARD_FORM
- zeta_C nabla_alpha Q^alpha_mu     cubic principal variation       EXPLICIT_DERIVED_FORM
                  zeta_C A_C_mu u-dependence of STF projector EXACT_FUNCTIONAL_DEFINITION
                2 lambda_u u_mu               unit constraint                    EXPLICIT
             sum_A eta_A e_A_mu             u.e_A constraints                    EXPLICIT
G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED = True
G28_CUBIC_VECTOR_ALGEBRAIC_REMAINDER_COMPONENT_EXPANDED = False


## 1.5 EOM métrique

Pour éviter toute ambiguïté de convention sur un pseudo-tenseur intermédiaire, on définit directement le tenseur d'Euler métrique :

\[
\boxed{
\mathcal E^{(g)}_{\mu\nu}
:=
\frac{16\pi G_0}{\sqrt{-g}}
\frac{\delta S_{\rm cand}}{\delta g^{\mu\nu}}
=0.
}
\]

Il se décompose exactement en :

\[
\mathcal E^{(g)}_{\mu\nu}
=
G_{\mu\nu}
+\mathcal H^{(u)}_{\mu\nu}
+\mathcal H^{(C)}_{\mu\nu}
+\mathcal H^{(\mathrm{con})}_{\mu\nu}.
\]

Le secteur cubique est défini sans circularité par :

\[
\boxed{
\mathcal H^{(C)}_{\mu\nu}
=
\frac{1}{\sqrt{-g}}
\frac{\delta}
{\delta g^{\mu\nu}}
\left[
\int d^4x\sqrt{-g}
\frac{\zeta_C}{2}\mathcal O_C
\right].
}
\]

Cette définition est exacte, mais **n'est pas encore l'expansion composante finale** : la métrique intervient dans \(\sqrt{-g}\), dans les indices, dans \(h_{\mu\nu}\), dans \(\sigma_{\mu\nu}\) et dans la connexion de Levi-Civita.

Ainsi `.3.3.28` matérialise le système variationnel complet, mais ne prétend pas que le stress cubique a déjà été réduit à une expression composante minimale.

In [6]:
metric_eom_ledger = pd.DataFrame([
    {"sector":"Einstein", "Euler_object":"G_mn", "status":"EXPLICIT"},
    {"sector":"EA vector", "Euler_object":"H_u_mn", "status":"ESTABLISHED_BENCHMARK_FORM_AVAILABLE"},
    {"sector":"cubic triad", "Euler_object":"H_C_mn", "status":"EXACT_FUNCTIONAL_DERIVATIVE_DEFINED"},
    {"sector":"constraints", "Euler_object":"H_con_mn", "status":"EXACT_FUNCTIONAL_DERIVATIVE_DEFINED"},
])

G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED = len(metric_eom_ledger) == 4
G28_FULL_COMPONENT_CUBIC_METRIC_STRESS_EXPANDED = False
G28_FULL_ARBITRARY_4D_COMPONENT_EOM_CLOSED = False

assert G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED
assert not G28_FULL_COMPONENT_CUBIC_METRIC_STRESS_EXPANDED
assert not G28_FULL_ARBITRARY_4D_COMPONENT_EOM_CLOSED

print(metric_eom_ledger.to_string(index=False))
print("G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED =",
      G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED)
print("G28_FULL_COMPONENT_CUBIC_METRIC_STRESS_EXPANDED =",
      G28_FULL_COMPONENT_CUBIC_METRIC_STRESS_EXPANDED)
print("G28_FULL_ARBITRARY_4D_COMPONENT_EOM_CLOSED =",
      G28_FULL_ARBITRARY_4D_COMPONENT_EOM_CLOSED)

     sector Euler_object                               status
   Einstein         G_mn                             EXPLICIT
  EA vector       H_u_mn ESTABLISHED_BENCHMARK_FORM_AVAILABLE
cubic triad       H_C_mn  EXACT_FUNCTIONAL_DERIVATIVE_DEFINED
constraints     H_con_mn  EXACT_FUNCTIONAL_DERIVATIVE_DEFINED
G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED = True
G28_FULL_COMPONENT_CUBIC_METRIC_STRESS_EXPANDED = False
G28_FULL_ARBITRARY_4D_COMPONENT_EOM_CLOSED = False


# Niveau 2 — Physique établie

## 2.1 Limite Einstein-æther comme benchmark, pas comme destination imposée

Le secteur \(\mathcal L_u\) à quatre couplages a déjà été identifié dans `.3.3.21` à la base quadratique Einstein-æther adaptée à la convention \((-+++)\).

Référence de contrôle déjà utilisée dans la chaîne :

- Foster & Jacobson, *Phys. Rev. D* **73**, 064015 (2006), arXiv:gr-qc/0509083.

Le test ici est uniquement :

\[
\zeta_C\to0.
\]

Alors :

\[
S_{\rm cand}\to S_{\rm EA}
\]

pour le secteur \(\{g_{\mu\nu},u^\mu\}\), et le trièdre perd son potentiel d'orientation.

C'est un **benchmark de limite**, pas une preuve de nouvelle physique GVH.

In [7]:
# Exact algebraic limit tests.
zeta_sym, c13_sym, sigma2_sym = sp.symbols(
    "zeta_C c13 sigma2", real=True
)

L_shear_aligned = -c13_sym*sigma2_sym + zeta_sym*sp.Rational(1,2)*sigma2_sym
c13_eff = sp.simplify(c13_sym-zeta_sym/2)

G28_ALIGNED_BRANCH_C13_SHIFT_PASS = (
    sp.simplify(L_shear_aligned + c13_eff*sigma2_sym) == 0
)
G28_ZETA_ZERO_RECOVERS_EA_SHEAR = (
    sp.simplify(L_shear_aligned.subs(zeta_sym,0) + c13_sym*sigma2_sym) == 0
)

# Orientation equation zeta*(sA-sB)*MAB=0 vanishes identically at zeta=0.
orientation_eqs = [
    zeta_sym*(s1-s2)*M12,
    zeta_sym*(s1-s3)*M13,
    zeta_sym*(s2-s3)*M23,
]
G28_ZETA_ZERO_TRIAD_ORIENTATION_DECOUPLES = all(
    sp.simplify(eq.subs(zeta_sym,0)) == 0
    for eq in orientation_eqs
)

G28_ESTABLISHED_EA_LIMIT_BENCHMARK_PASS = all([
    G28_ALIGNED_BRANCH_C13_SHIFT_PASS,
    G28_ZETA_ZERO_RECOVERS_EA_SHEAR,
    G28_ZETA_ZERO_TRIAD_ORIENTATION_DECOUPLES,
])

assert G28_ESTABLISHED_EA_LIMIT_BENCHMARK_PASS

print("c13_eff =", c13_eff)
print("G28_ALIGNED_BRANCH_C13_SHIFT_PASS =", G28_ALIGNED_BRANCH_C13_SHIFT_PASS)
print("G28_ZETA_ZERO_RECOVERS_EA_SHEAR =", G28_ZETA_ZERO_RECOVERS_EA_SHEAR)
print("G28_ZETA_ZERO_TRIAD_ORIENTATION_DECOUPLES =",
      G28_ZETA_ZERO_TRIAD_ORIENTATION_DECOUPLES)
print("G28_ESTABLISHED_EA_LIMIT_BENCHMARK_PASS =",
      G28_ESTABLISHED_EA_LIMIT_BENCHMARK_PASS)

c13_eff = c13 - zeta_C/2
G28_ALIGNED_BRANCH_C13_SHIFT_PASS = True
G28_ZETA_ZERO_RECOVERS_EA_SHEAR = True
G28_ZETA_ZERO_TRIAD_ORIENTATION_DECOUPLES = True
G28_ESTABLISHED_EA_LIMIT_BENCHMARK_PASS = True


# Niveau 3 — Audit P/D après ouverture des EOM

Le fait d'avoir maintenant des EOM ne suffit **pas** à promouvoir automatiquement les paramètres du benchmark `.3.3.27.2`.

Les trois entrées critiques restent :

\[
\omega_{10},\qquad
\omega_{21},\qquad
u_2.
\]

Pour les promouvoir en `EOM_DERIVED`, il faudrait :

1. choisir un ansatz de solution relié sans ambiguïté au benchmark line \(\to\) circle \(\to\) helix ;
2. insérer cet ansatz dans les EOM ;
3. montrer que les équations réduites sélectionnent ces taux ou des relations entre eux ;
4. distinguer les constantes d'intégration et conditions initiales des constantes véritablement fixées par la théorie.

En revanche, certaines nouvelles quantités deviennent réellement **EOM-constrained** :

- les orientations du trièdre ;
- les multiplicateurs de contraintes ;
- les relations de branche \((s_A-s_B)M_{AB}=0\).

Les échelles \(L_*,T_*\) de `.3.3.27.4` ne sont pas fixées par la seule écriture variationnelle.

In [8]:
pd_after_eom = pd.DataFrame([
    {"object":"triad orientation q^a", "previous":"AUXILIARY_UNSELECTED", "now":"EOM_CONSTRAINED_BRANCH", "derived_numeric_value":False},
    {"object":"eta_A, eta_AB", "previous":"MULTIPLIERS", "now":"ALGEBRAICALLY_DERIVABLE_ON_BRANCH", "derived_numeric_value":False},
    {"object":"omega10", "previous":"PRESCRIBED_NOT_DERIVED", "now":"PRESCRIBED_NOT_DERIVED", "derived_numeric_value":False},
    {"object":"omega21", "previous":"PRESCRIBED_NOT_DERIVED", "now":"PRESCRIBED_NOT_DERIVED", "derived_numeric_value":False},
    {"object":"u2", "previous":"PRESCRIBED_NOT_DERIVED", "now":"PRESCRIBED_NOT_DERIVED", "derived_numeric_value":False},
    {"object":"L_star", "previous":"FREE_NOT_DERIVED", "now":"FREE_NOT_DERIVED", "derived_numeric_value":False},
    {"object":"T_star", "previous":"FREE_NOT_DERIVED", "now":"FREE_NOT_DERIVED", "derived_numeric_value":False},
    {"object":"chi_helix", "previous":"COMPUTABLE_NOT_EOM_DERIVED", "now":"COMPUTABLE_NOT_EOM_DERIVED", "derived_numeric_value":False},
])

critical_names = {"omega10","omega21","u2"}
critical_rows = pd_after_eom[pd_after_eom["object"].isin(critical_names)]

G28_ANY_INTERNAL_P_TO_D_PROMOTION = True
G28_CRITICAL_KINEMATIC_INPUTS_DERIVED = bool(
    (critical_rows["now"] == "EOM_DERIVED").all()
)
G28_ABSOLUTE_SI_SCALE_DERIVED = False
G28_SCALE_FREE_HELIX_RATIO_EOM_DERIVED = False

assert G28_ANY_INTERNAL_P_TO_D_PROMOTION
assert not G28_CRITICAL_KINEMATIC_INPUTS_DERIVED
assert not G28_ABSOLUTE_SI_SCALE_DERIVED
assert not G28_SCALE_FREE_HELIX_RATIO_EOM_DERIVED

print(pd_after_eom.to_string(index=False))
print("G28_CRITICAL_KINEMATIC_INPUTS_DERIVED =",
      G28_CRITICAL_KINEMATIC_INPUTS_DERIVED)
print("G28_ABSOLUTE_SI_SCALE_DERIVED =", G28_ABSOLUTE_SI_SCALE_DERIVED)
print("G28_SCALE_FREE_HELIX_RATIO_EOM_DERIVED =",
      G28_SCALE_FREE_HELIX_RATIO_EOM_DERIVED)

               object                   previous                               now  derived_numeric_value
triad orientation q^a       AUXILIARY_UNSELECTED            EOM_CONSTRAINED_BRANCH                  False
        eta_A, eta_AB                MULTIPLIERS ALGEBRAICALLY_DERIVABLE_ON_BRANCH                  False
              omega10     PRESCRIBED_NOT_DERIVED            PRESCRIBED_NOT_DERIVED                  False
              omega21     PRESCRIBED_NOT_DERIVED            PRESCRIBED_NOT_DERIVED                  False
                   u2     PRESCRIBED_NOT_DERIVED            PRESCRIBED_NOT_DERIVED                  False
               L_star           FREE_NOT_DERIVED                  FREE_NOT_DERIVED                  False
               T_star           FREE_NOT_DERIVED                  FREE_NOT_DERIVED                  False
            chi_helix COMPUTABLE_NOT_EOM_DERIVED        COMPUTABLE_NOT_EOM_DERIVED                  False
G28_CRITICAL_KINEMATIC_INPUTS_DERIVED = False


## 3.1 Témoin numérique indépendant de la variation d'orientation

On choisit un cisaillement STF numérique :

\[
M=
\begin{pmatrix}
0.8&0.12&-0.07\\
0.12&-0.3&0.09\\
-0.07&0.09&-0.5
\end{pmatrix},
\qquad
\mathrm{tr}\,M=0.
\]

On compare :

1. les dérivées analytiques

\[
\frac{d\mathcal O_C}{d\theta_{12}}
=
4M_{12}(M_{11}-M_{22}),
\]

et analogues ;

2. une différence centrale avec

\[
\epsilon=10^{-6}.
\]

Tolérance :

\[
\boxed{5\times10^{-9}}
\]

sur l'erreur absolue maximale.

Cette tolérance est volontairement bien au-dessus de l'erreur d'arrondi attendue pour une différence centrale à ce pas, tout en restant assez stricte pour détecter un facteur ou un signe erroné.

In [9]:
M0 = np.array([
    [ 0.8,  0.12, -0.07],
    [ 0.12,-0.3,   0.09],
    [-0.07, 0.09, -0.5 ],
], dtype=float)

G28_NUMERIC_STF_TRACE_PASS = abs(np.trace(M0)) < 1e-15

def R12_num(t):
    return np.array([
        [np.cos(t),-np.sin(t),0.0],
        [np.sin(t), np.cos(t),0.0],
        [0.0,0.0,1.0],
    ])

def R13_num(t):
    return np.array([
        [np.cos(t),0.0,-np.sin(t)],
        [0.0,1.0,0.0],
        [np.sin(t),0.0,np.cos(t)],
    ])

def R23_num(t):
    return np.array([
        [1.0,0.0,0.0],
        [0.0,np.cos(t),-np.sin(t)],
        [0.0,np.sin(t), np.cos(t)],
    ])

def OC_num(R):
    Mr = R.T @ M0 @ R
    return float(np.sum(np.diag(Mr)**2))

eps = 1e-6
analytic = np.array([
    4*M0[0,1]*(M0[0,0]-M0[1,1]),
    4*M0[0,2]*(M0[0,0]-M0[2,2]),
    4*M0[1,2]*(M0[1,1]-M0[2,2]),
])

numeric = np.array([
    (OC_num(R(eps))-OC_num(R(-eps)))/(2*eps)
    for R in (R12_num,R13_num,R23_num)
])

errors = np.abs(numeric-analytic)
G28_ORIENTATION_NUMERIC_ABS_TOL = 5e-9
G28_ORIENTATION_NUMERIC_MAX_ERROR = float(np.max(errors))
G28_ORIENTATION_NUMERIC_WITNESS_PASS = (
    G28_NUMERIC_STF_TRACE_PASS
    and G28_ORIENTATION_NUMERIC_MAX_ERROR < G28_ORIENTATION_NUMERIC_ABS_TOL
)

assert G28_ORIENTATION_NUMERIC_WITNESS_PASS

print("analytic =", analytic)
print("numeric  =", numeric)
print("errors   =", errors)
print("max error =", G28_ORIENTATION_NUMERIC_MAX_ERROR)
print("tolerance =", G28_ORIENTATION_NUMERIC_ABS_TOL)
print("G28_ORIENTATION_NUMERIC_WITNESS_PASS =",
      G28_ORIENTATION_NUMERIC_WITNESS_PASS)

analytic = [ 0.528 -0.364  0.072]
numeric  = [ 0.528 -0.364  0.072]
errors   = [1.10662146e-10 3.04510306e-11 3.78976156e-11]
max error = 1.1066214611332725e-10
tolerance = 5e-09
G28_ORIENTATION_NUMERIC_WITNESS_PASS = True


# Niveau 4 — Verdict scientifique

Le résultat de `.3.3.28` doit être formulé à deux niveaux.

### Ce qui est réellement dérivé

\[
\boxed{
\mathcal E^{(e_A)}_\mu=0
}
\]

est explicite, avec la condition d'orientation :

\[
\boxed{
\zeta_C(s_A-s_B)M_{AB}=0.
}
\]

L'EOM \(u^\mu\) est matérialisée sous forme variationnelle exacte avec son courant principal cubique, et l'EOM métrique est définie comme dérivée fonctionnelle exacte.

### Ce qui ne doit pas être sur-promu

L'expansion composante complète de :

\[
\mathcal H^{(C)}_{\mu\nu}
\]

et du reste algébrique :

\[
\mathcal A^{(C)}_\mu
\]

n'est pas encore terminée.

Surtout :

\[
\boxed{
\omega_{10},\omega_{21},u_2
\text{ ne sont toujours pas EOM-derived.}
}
\]

Donc :

\[
\boxed{
\texttt{NEW\_GVH\_PHYSICS\_VALIDATED=False}.
}
\]

La séquence rigoureuse suivante reste dans la branche `.3.3.28.x` : développer complètement les termes cubiques puis réduire les EOM sur un ansatz minimal relié au benchmark hélicoïdal, avant d'ouvrir une nouvelle étape principale.

In [10]:
LEVEL1 = "CONSTRAINED_ACTION_AND_EULER_DERIVATIVES_MATERIALIZED"
LEVEL2 = "EA_LIMIT_AND_AUXILIARY_TRIAD_BENCHMARK_CONSISTENT"
LEVEL3 = "NUMERIC_ORIENTATION_VARIATION_WITNESS"
LEVEL4 = "BLOCKED_FULL_COMPONENT_CUBIC_STRESS_AND_REDUCED_BENCHMARK_SOLUTION"

G28_FULL_VARIATIONAL_SYSTEM_MATERIALIZED = all([
    G28_CONSTRAINED_ACTION_MATERIALIZED,
    G28_TRIAD_ORIENTATION_EOM_EXPLICIT,
    G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED,
    G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED,
])

G28_EOM_DERIVATION_FULLY_COMPONENT_EXPANDED = all([
    G28_FULL_COMPONENT_CUBIC_METRIC_STRESS_EXPANDED,
    G28_CUBIC_VECTOR_ALGEBRAIC_REMAINDER_COMPONENT_EXPANDED,
    G28_FULL_ARBITRARY_4D_COMPONENT_EOM_CLOSED,
])

G28_NEW_GVH_PHYSICS_VALIDATED = False
G28_REAL_DATA_READY = False
G28_HELIX_EOM_SOLUTION_DERIVED = False

G28_LOCAL_AUDIT_PASS = all([
    G28_UPSTREAM_PROVENANCE_PASS,
    G28_CONSTRAINED_ACTION_MATERIALIZED,
    G28_ORIENTATION_FIRST_VARIATION_EXACT_PASS,
    G28_TRIAD_EOM_PROJECTED_EXACT_PASS,
    G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED,
    G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED,
    G28_ESTABLISHED_EA_LIMIT_BENCHMARK_PASS,
    G28_ORIENTATION_NUMERIC_WITNESS_PASS,
    not G28_CRITICAL_KINEMATIC_INPUTS_DERIVED,
    not G28_ABSOLUTE_SI_SCALE_DERIVED,
    not G28_EOM_DERIVATION_FULLY_COMPONENT_EXPANDED,
    not G28_NEW_GVH_PHYSICS_VALIDATED,
])

G28_OBSTRUCTIONS = [
    "FULL_COMPONENT_CUBIC_METRIC_STRESS_NOT_EXPANDED",
    "CUBIC_VECTOR_PROJECTOR_ALGEBRAIC_REMAINDER_NOT_COMPONENT_EXPANDED",
    "NO_REDUCED_EOM_SOLUTION_MAPPED_TO_OMEGA10_OMEGA21_U2",
    "ABSOLUTE_SI_SCALES_NOT_THEORETICALLY_DERIVED",
    "CUBIC_TRIAD_REMAINS_AUXILIARY_WITH_ZERO_OWN_KINETIC_TERM",
    "ON_SHELL_GVH_DISTINCTNESS_NOT_ESTABLISHED",
]

G28_NEXT_AUTHORIZED = (
    "0.3.2.7.3.7.3.3.28.1_"
    "EXPAND_CUBIC_METRIC_VECTOR_EOM_AND_REDUCE_MINIMAL_HELIX_ANSATZ"
)

assert G28_LOCAL_AUDIT_PASS

artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST",
    "upstream": UPSTREAM,
    "open_refs": G28_BLOCKED_OPEN_REFS,
    "levels": {
        "LEVEL1": LEVEL1,
        "LEVEL2": LEVEL2,
        "LEVEL3": LEVEL3,
        "LEVEL4": LEVEL4,
    },
    "eom": {
        "constrained_action_materialized": G28_CONSTRAINED_ACTION_MATERIALIZED,
        "triad_orientation_eom_explicit": G28_TRIAD_ORIENTATION_EOM_EXPLICIT,
        "vector_functional_derivative_materialized": G28_VECTOR_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED,
        "metric_functional_derivative_materialized": G28_METRIC_EOM_FUNCTIONAL_DERIVATIVE_MATERIALIZED,
        "full_component_cubic_metric_stress_expanded": G28_FULL_COMPONENT_CUBIC_METRIC_STRESS_EXPANDED,
        "full_arbitrary_4D_component_eom_closed": G28_FULL_ARBITRARY_4D_COMPONENT_EOM_CLOSED,
    },
    "pd": {
        "critical_kinematic_inputs_derived": G28_CRITICAL_KINEMATIC_INPUTS_DERIVED,
        "absolute_SI_scale_derived": G28_ABSOLUTE_SI_SCALE_DERIVED,
        "scale_free_helix_ratio_eom_derived": G28_SCALE_FREE_HELIX_RATIO_EOM_DERIVED,
    },
    "numeric": {
        "orientation_variation_witness_pass": G28_ORIENTATION_NUMERIC_WITNESS_PASS,
        "max_abs_error": G28_ORIENTATION_NUMERIC_MAX_ERROR,
        "abs_tol": G28_ORIENTATION_NUMERIC_ABS_TOL,
    },
    "locks": {
        "helix_eom_solution_derived": G28_HELIX_EOM_SOLUTION_DERIVED,
        "real_data_ready": G28_REAL_DATA_READY,
        "new_GVH_physics_validated": G28_NEW_GVH_PHYSICS_VALIDATED,
    },
    "verdict": {
        "G28_FULL_VARIATIONAL_SYSTEM_MATERIALIZED": G28_FULL_VARIATIONAL_SYSTEM_MATERIALIZED,
        "G28_EOM_DERIVATION_FULLY_COMPONENT_EXPANDED": G28_EOM_DERIVATION_FULLY_COMPONENT_EXPANDED,
        "G28_LOCAL_AUDIT_PASS": G28_LOCAL_AUDIT_PASS,
        "obstructions": G28_OBSTRUCTIONS,
        "next_authorized": G28_NEXT_AUTHORIZED,
    },
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")

print("LEVEL1 =", LEVEL1)
print("LEVEL2 =", LEVEL2)
print("LEVEL3 =", LEVEL3)
print("LEVEL4 =", LEVEL4)
print("G28_FULL_VARIATIONAL_SYSTEM_MATERIALIZED =", G28_FULL_VARIATIONAL_SYSTEM_MATERIALIZED)
print("G28_EOM_DERIVATION_FULLY_COMPONENT_EXPANDED =", G28_EOM_DERIVATION_FULLY_COMPONENT_EXPANDED)
print("G28_CRITICAL_KINEMATIC_INPUTS_DERIVED =", G28_CRITICAL_KINEMATIC_INPUTS_DERIVED)
print("G28_ABSOLUTE_SI_SCALE_DERIVED =", G28_ABSOLUTE_SI_SCALE_DERIVED)
print("G28_NEW_GVH_PHYSICS_VALIDATED =", G28_NEW_GVH_PHYSICS_VALIDATED)
print("G28_LOCAL_AUDIT_PASS =", G28_LOCAL_AUDIT_PASS)
print("G28_OBSTRUCTIONS =", G28_OBSTRUCTIONS)
print("G28_NEXT_AUTHORIZED =", G28_NEXT_AUTHORIZED)
print("G28 artifact =", artifact_path)

LEVEL1 = CONSTRAINED_ACTION_AND_EULER_DERIVATIVES_MATERIALIZED
LEVEL2 = EA_LIMIT_AND_AUXILIARY_TRIAD_BENCHMARK_CONSISTENT
LEVEL3 = NUMERIC_ORIENTATION_VARIATION_WITNESS
LEVEL4 = BLOCKED_FULL_COMPONENT_CUBIC_STRESS_AND_REDUCED_BENCHMARK_SOLUTION
G28_FULL_VARIATIONAL_SYSTEM_MATERIALIZED = True
G28_EOM_DERIVATION_FULLY_COMPONENT_EXPANDED = False
G28_CRITICAL_KINEMATIC_INPUTS_DERIVED = False
G28_ABSOLUTE_SI_SCALE_DERIVED = False
G28_NEW_GVH_PHYSICS_VALIDATED = False
G28_LOCAL_AUDIT_PASS = True
G28_OBSTRUCTIONS = ['FULL_COMPONENT_CUBIC_METRIC_STRESS_NOT_EXPANDED', 'CUBIC_VECTOR_PROJECTOR_ALGEBRAIC_REMAINDER_NOT_COMPONENT_EXPANDED', 'NO_REDUCED_EOM_SOLUTION_MAPPED_TO_OMEGA10_OMEGA21_U2', 'ABSOLUTE_SI_SCALES_NOT_THEORETICALLY_DERIVED', 'CUBIC_TRIAD_REMAINS_AUXILIARY_WITH_ZERO_OWN_KINETIC_TERM', 'ON_SHELL_GVH_DISTINCTNESS_NOT_ESTABLISHED']
G28_NEXT_AUTHORIZED = 0.3.2.7.3.7.3.3.28.1_EXPAND_CUBIC_METRIC_VECTOR_EOM_AND_REDUCE_MINIMAL_HELIX_ANSATZ
G28 artifact = /content/gvh_exports/gvh_0.3.2.7.3.